In [ ]:
import base64
import json
import logging
import os
import time
import requests
import pandas as pd

logger = logging.getLogger(__name__)
logger.setLevel(logging.DEBUG)
dh = logging.FileHandler("debug.log", encoding="utf-8")
dh.setLevel(logging.DEBUG)
eh = logging.FileHandler("error.log", encoding="utf-8")
eh.setLevel(logging.ERROR)
formatter = logging.Formatter("%(asctime)s-%(name)s-%(levelname)s-%(message)s")
dh.setFormatter(formatter)
eh.setFormatter(formatter)
logger.addHandler(eh)
logger.addHandler(dh)

H = {"User-Agent": "hse-bbi-237-student", "Accept": "application/vnd.github+json"}
# https://yandex.ru/search/?text=как+добавить+токен+в+google+colab&clid=1955453&win=723&lr=213
# https://colab.research.google.com/github/google-gemini/cookbook/blob/main/quickstarts/Authentication_with_OAuth.ipynb
try:
    from google.colab import userdata
    os.environ["GITHUB_TOKEN"] = userdata.get("GITHUB_TOKEN")
except ImportError:
    pass
tk = os.environ.get("GITHUB_TOKEN")
if tk:
    H["Authorization"] = f"Bearer {tk}"


def get_json(url, params=None):
    n_stop = 0
    while True:
        logger.debug(f"GET_JSON {url} params={params}")
        r = requests.get(url, params=params, headers=H)
        # Ошибки подбирались потом и кровью. Обе ошибки встречали при скачке
        if r.status_code in (403, 429):
            logger.warning(f"GET_JSON rate_limit {url} {r.status_code} {r!r}")
            n_stop += 1
            if n_stop >= 2:
                return r
            time.sleep(60)
            continue
        # time.sleep(0.5)
        return r

In [ ]:
seen = set()
users = []
req_n = 0
p = 1
while p <= 10:
    logger.debug(f"SEARCH_USERS page={p}")
    r = get_json(
        "https://api.github.com/search/users",
        params={"q": "location:Russia", "per_page": 100, "page": p},
    )
    try:
        b = r.json()["items"]
    except (ValueError, KeyError, TypeError):
        break
    if not b:
        break
    for u in b:
        try:
            uid = u["id"]
            lg = u["login"]
            if uid in seen:
                continue
            seen.add(uid)
            ru = get_json(f"https://api.github.com/users/{lg}")
            req_n += 1
            if ru.status_code == 200:
                users.append(ru.json())
            else:
                users.append(u)
            with open("users.json", "w", encoding="utf-8") as f:
                json.dump(users, f, ensure_ascii=False, indent=2)
            if req_n % 100 == 0:
                logger.debug(f"USERS_PROGRESS {req_n} {len(users)}")
        except (KeyError, TypeError, ValueError):
            continue
    if len(b) < 100:
        break
    p += 1

df_users = pd.json_normalize(users, sep="_")
df_users

In [ ]:
# with open("users.json", encoding="utf-8") as f:
#     users = json.load(f)
# df_users = pd.json_normalize(users, sep="_")

In [ ]:
rows = []
open("repos.jsonl", "w", encoding="utf-8").close()
req_n = 0
for k in df_users.index:
    lg = df_users.at[k, "login"]
    pr = df_users.at[k, "public_repos"]
    if pd.notna(pr) and pr >= 30:
        continue
    p = 1
    while True:
        rr = get_json(
            f"https://api.github.com/users/{lg}/repos",
            params={"per_page": 100, "page": p},
        )
        req_n += 1
        if req_n % 100 == 0:
            logger.debug(f"REPOS_REQ {req_n}")
        if rr.status_code != 200:
            break
        try:
            b = rr.json()
        except ValueError:
            break
        if not b:
            break
        for x in b:
            x["sample_user_login"] = lg
            rows.append(x)
            with open("repos.jsonl", "a", encoding="utf-8") as f:
                f.write(json.dumps(x, ensure_ascii=False) + "\n")
        if len(b) < 100:
            break
        p += 1

df_repos = pd.json_normalize(rows, sep="_")
df_repos

In [ ]:
# rows = []
# with open("repos.jsonl", encoding="utf-8") as f:
#     for line in f:
#         rows.append(json.loads(line))
# df_repos = pd.json_normalize(rows, sep="_")

In [ ]:
def dec(s):
    if not s:
        return None
    return base64.b64decode(s.replace("\n", "")).decode("utf-8", errors="replace")

df_repos_enriched = df_repos.copy()
prev = {}
if os.path.isfile("repos_enriched.jsonl"):
    with open("repos_enriched.jsonl", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                d = json.loads(line)
            except json.JSONDecodeError:
                continue
            k = d.get("full_name")
            if k:
                prev[k] = d
df_repos_enriched["readme"] = None
df_repos_enriched["lang_bytes"] = None
df_repos_enriched["lang_pct"] = None
req_n = 0
for i in df_repos_enriched.index:
    fn = df_repos_enriched.at[i, "full_name"]
    if fn in prev:
        o = prev[fn]
        df_repos_enriched.at[i, "readme"] = o.get("readme")
        df_repos_enriched.at[i, "lang_bytes"] = o.get("lang_bytes")
        df_repos_enriched.at[i, "lang_pct"] = o.get("lang_pct")
        continue
    try:
        r = get_json(f"https://api.github.com/repos/{fn}/readme")
        if r.status_code == 200:
            j = r.json()
            df_repos_enriched.at[i, "readme"] = dec(j.get("content", None))
        r = get_json(f"https://api.github.com/repos/{fn}/languages")
        if r.status_code == 200:
            lb = r.json()
            df_repos_enriched.at[i, "lang_bytes"] = lb
            t = sum(lb.values())
            if t > 0:
                df_repos_enriched.at[i, "lang_pct"] = {k: 100.0 * v / t for k, v in lb.items()}
            else:
                df_repos_enriched.at[i, "lang_pct"] = {}
        else:
            df_repos_enriched.at[i, "lang_bytes"] = None
            df_repos_enriched.at[i, "lang_pct"] = None
    except (ValueError, AttributeError, TypeError):
        df_repos_enriched.at[i, "readme"] = None
        df_repos_enriched.at[i, "lang_bytes"] = None
        df_repos_enriched.at[i, "lang_pct"] = None
    with open("repos_enriched.jsonl", "a", encoding="utf-8") as f:
        f.write(df_repos_enriched.loc[[i]].to_json(orient="records", lines=True) + "\n")
    req_n += 1
    if req_n % 100 == 0:
        logger.debug(f"ENRICH_REQ {req_n}")

df_repos_enriched

In [ ]:
# with open("repos_enriched.jsonl", encoding="utf-8") as f:
#     rows = [json.loads(line) for line in f]
# df_repos_enriched = pd.json_normalize(rows, sep="_")

In [ ]:
cols_r = [
    "sample_user_login", "id", "name", "full_name", "html_url",
    "description", "fork", "private", "archived", "disabled",
    "owner_login", "owner_type",
    "language", "topics", "license_spdx_id", "license_name",
    "stargazers_count", "forks_count", "watchers_count", "open_issues_count",
    "size", "created_at", "updated_at", "pushed_at",
    "default_branch", "visibility",
    "readme",
    "lang_bytes", "lang_pct",
]
df_repos_filtered = df_repos_enriched[[c for c in cols_r if c in df_repos_enriched.columns]]
df_repos_filtered

In [ ]:
cols_u = [
    "login", "id", "type", "html_url", "site_admin", "user_view_type",
    "name", "bio", "location", "company", "blog", "hireable",
    "followers", "following", "public_repos", "public_gists",
    "created_at", "updated_at", "twitter_username",
]
df_users_filtered = df_users[[c for c in cols_u if c in df_users.columns]]
df_users_filtered

In [ ]:
df_users_filtered.to_csv("users_filtered.csv")
df_repos_filtered.to_csv("repos_filtered.csv")